# Fake News Classification with TFX Pipeline

Proyek ini membangun pipeline _end-to-end_ untuk klasifikasi berita palsu menggunakan **TensorFlow Extended (TFX)**. Seluruh komponen TFX dijalankan menggunakan **InteractiveContext** sehingga cocok untuk eksperimen interaktif di Jupyter Notebook.

---

## 1. Dataset

**Dataset**: [Fake and Real News Dataset](https://www.kaggle.com/datasets/clmentbisaillon/fake-and-real-news-dataset)  
**Sumber**: Kaggle — Clment Bisaillon  

Dataset berisi ~44.000 artikel berita dalam bahasa Inggris dengan dua label:

| Label | Deskripsi |
|-------|-----------|
| FAKE  | Berita palsu |
| REAL  | Berita asli  |

**Fitur dalam dataset**:
- `title` — judul artikel
- `text` — isi artikel
- `subject` — kategori topik
- `date` — tanggal publikasi

Dataset tersedia dalam dua file CSV: `Fake.csv` (berita palsu) dan `True.csv` (berita asli).

## 2. Masalah

Penyebaran berita palsu (_fake news_) di platform digital semakin marak dan sulit dibedakan secara manual oleh pembaca. Dampaknya meliputi:
- Misinformasi publik
- Polarisasi opini masyarakat
- Kerugian bagi individu atau organisasi

Dibutuhkan sistem otomatis yang dapat mengklasifikasikan berita sebagai **palsu atau asli** secara cepat dan akurat.

## 3. Solusi Machine Learning

Solusi yang dibangun adalah **pipeline ML end-to-end** menggunakan **TensorFlow Extended (TFX)** yang mencakup 9 komponen:

1. **ExampleGen** — Membaca dataset CSV dan mengonversinya ke TFRecord
2. **StatisticsGen** — Menghitung statistik data
3. **SchemaGen** — Membuat skema data otomatis
4. **ExampleValidator** — Mendeteksi anomali dalam data
5. **Transform** — Preprocessing teks dan encoding label
6. **Trainer** — Melatih model neural network
7. **Evaluator** — Mengevaluasi performa model dan memberikan _blessing_
8. **Resolver** — Memilih model terbaik yang sudah di-_bless_
9. **Pusher** — Menyimpan model siap _serving_

Pipeline dijalankan menggunakan **InteractiveContext** untuk pengembangan interaktif.

## 4. Setup Environment

In [ ]:
# Pastikan sudah install requirements.txt dari terminal:
#   pip install -r requirements.txt
#
# Catatan Penting:
# - Wajib menggunakan Python 3.10
# - TFX 1.15.x TIDAK support Python 3.11
# - Gunakan virtual environment terpisah

import os
import sys
import json
import math
import shutil
import tempfile
import warnings
warnings.filterwarnings('ignore')

if sys.version_info[:2] != (3, 10):
    raise RuntimeError(
        f'Gunakan Python 3.10 (sekarang: {sys.version}). '
        f'TFX 1.15.x hanya support Python >=3.9,<3.11'
    )

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
import tensorflow_data_validation as tfdv
import tensorflow_transform as tft
import tensorflow_model_analysis as tfma

from tfx import v1 as tfx
from tfx.orchestration.experimental.interactive import InteractiveContext
from tfx.components import CsvExampleGen, StatisticsGen, SchemaGen
from tfx.components import ExampleValidator, Transform, Trainer, Evaluator, Pusher
from tfx.dsl.components.common.resolver import Resolver
from tfx.dsl.input_resolution.strategies.latest_blessed_model_strategy import LatestBlessedModelStrategy
from tfx.types.standard_artifacts import Model, ModelBlessing
from tfx.types.channel import Channel
from tfx.proto import example_gen_pb2, trainer_pb2, pusher_pb2, evaluator_pb2

print(f'Python version: {sys.version}')
print(f'TensorFlow version: {tf.__version__}')
print(f'TFX version: {tfx.__version__}')

## 5. Persiapan Data

Download dataset dari Kaggle menggunakan `kagglehub`, lalu gabungkan file `Fake.csv` dan `True.csv` menjadi satu dataset dengan label.

In [ ]:
import kagglehub

print('Downloading dataset from Kaggle...')
dataset_path = kagglehub.dataset_download('clmentbisaillon/fake-and-real-news-dataset')
print(f'Dataset downloaded to: {dataset_path}')

df_fake = pd.read_csv(os.path.join(dataset_path, 'Fake.csv'))
df_true = pd.read_csv(os.path.join(dataset_path, 'True.csv'))

print(f'Fake news articles: {len(df_fake)}')
print(f'Real news articles: {len(df_true)}')

In [ ]:
df_fake['label'] = 'FAKE'
df_true['label'] = 'REAL'

df = pd.concat([df_fake, df_true], ignore_index=True)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
df = df[['title', 'text', 'label']]

print(f'Total samples: {len(df)}')
print(f'Label distribution:')
print(df['label'].value_counts())
df.head()

In [ ]:
train_size = int(len(df) * 0.8)
df_train = df[:train_size].reset_index(drop=True)
df_eval = df[train_size:].reset_index(drop=True)

print(f'Train samples: {len(df_train)}')
print(f'Eval samples: {len(df_eval)}')

DATA_DIR = './data'
os.makedirs(DATA_DIR, exist_ok=True)

df_train.to_csv(os.path.join(DATA_DIR, 'train.csv'), index=False)
df_eval.to_csv(os.path.join(DATA_DIR, 'eval.csv'), index=False)
print(f'Data saved to {DATA_DIR}/')

## 6. Metode Preprocessing Data

Preprocessing dilakukan oleh komponen **Transform** menggunakan `tensorflow-transform`:

| Langkah | Detail |
|---------|--------|
| 1. Penggabungan | Kolom `title` dan `text` digabung dengan spasi |
| 2. Lowercasing | Semua huruf diubah ke huruf kecil |
| 3. Cleaning | Karakter non-alfanumerik dihapus |
| 4. Whitespace | Spasi berlebih dihapus |
| 5. Label Encoding | `REAL` \u2192 1, `FAKE` \u2192 0 (int64) |

Hasil preprocessing adalah dua fitur:
- `text_xf` — teks yang sudah dibersihkan
- `label_xf` — label dalam format integer

Kode preprocessing ada di file `transform.py`.

## 7. TFX Pipeline

Pipeline dijalankan menggunakan **InteractiveContext**. Setiap komponen dieksekusi secara berurutan.

In [ ]:
PIPELINE_ROOT = os.path.join(tempfile.gettempdir(), 'tfx_pipeline')
os.makedirs(PIPELINE_ROOT, exist_ok=True)

context = InteractiveContext(
    pipeline_root=PIPELINE_ROOT,
    pipeline_name='fake-news-classification'
)

print(f'Pipeline root: {PIPELINE_ROOT}')

### 7.1 ExampleGen

Membaca dataset CSV (train.csv & eval.csv) dan mengonversinya ke format TFRecord.

In [ ]:
example_gen = CsvExampleGen(input_base=DATA_DIR)
context.run(example_gen)

artifact = example_gen.outputs['examples'].get()[0]
print(f'Examples URI: {artifact.uri}')

### 7.2 StatisticsGen

Menghitung statistik data untuk mendapatkan insight dan validasi kualitas data.

In [ ]:
statistics_gen = StatisticsGen(examples=example_gen.outputs['examples'])
context.run(statistics_gen)
context.show(statistics_gen.outputs['statistics'])

### 7.3 SchemaGen

Membuat skema data secara otomatis berdasarkan statistik yang dihasilkan.

In [ ]:
schema_gen = SchemaGen(statistics=statistics_gen.outputs['statistics'])
context.run(schema_gen)
context.show(schema_gen.outputs['schema'])

### 7.4 ExampleValidator

Mendeteksi anomali data berdasarkan skema yang telah dibuat.

In [ ]:
example_validator = ExampleValidator(
    statistics=statistics_gen.outputs['statistics'],
    schema=schema_gen.outputs['schema']
)
context.run(example_validator)
context.show(example_validator.outputs['anomalies'])

### 7.5 Transform

Preprocessing teks menggunakan modul `transform.py`. Membersihkan teks dan meng-encode label.

In [ ]:
transform = Transform(
    examples=example_gen.outputs['examples'],
    schema=schema_gen.outputs['schema'],
    module_file=os.path.abspath('transform.py')
)
context.run(transform)

## 8. Arsitektur Model

Model menggunakan arsitektur neural network untuk klasifikasi teks:

```
Input (string teks)
       \u2193
TextVectorization (max_tokens=15000, output_sequence_length=200)
       \u2193
Embedding (vocab_size=15000, embedding_dim=128)
       \u2193
GlobalAveragePooling1D
       \u2193
Dense (64, ReLU)
       \u2193
Dropout (0.5)
       \u2193
Dense (32, ReLU)
       \u2193
Dropout (0.3)
       \u2193
Dense (1, Sigmoid) \u2192 Output probabilitas (0=FAKE, 1=REAL)
```

**Penjelasan setiap layer:**
- **Input**: Menerima teks dalam bentuk string
- **TextVectorization**: Mengubah teks menjadi urutan integer berdasarkan vocabulary yang dipelajari dari data
- **Embedding**: Mempelajari representasi vektor dense (128 dimensi) untuk setiap kata
- **GlobalAveragePooling1D**: Merata-ratakan seluruh vektor kata menjadi satu vektor representasi
- **Dense (64, ReLU)**: Fully connected layer dengan 64 neuron dan aktivasi ReLU
- **Dropout (0.5)**: Regularisasi dengan mematikan 50% neuron secara acak
- **Dense (32, ReLU)**: Fully connected layer dengan 32 neuron
- **Dropout (0.3)**: Regularisasi tambahan dengan dropout 30%
- **Dense (1, Sigmoid)**: Output layer dengan aktivasi sigmoid untuk klasifikasi biner

Kode arsitektur model ada di file `trainer.py`.

### 7.6 Trainer

Melatih model menggunakan modul `trainer.py`. Model akan belajar mengklasifikasikan berita FAKE vs REAL.

In [ ]:
trainer = Trainer(
    module_file=os.path.abspath('trainer.py'),
    examples=transform.outputs['transformed_examples'],
    transform_graph=transform.outputs['transform_graph'],
    schema=schema_gen.outputs['schema'],
    train_args=trainer_pb2.TrainArgs(num_steps=0),
    eval_args=trainer_pb2.EvalArgs(num_steps=0)
)
context.run(trainer)

print('Training completed.')

## 9. Metrik Evaluasi

Model dievaluasi menggunakan metrik berikut:

| Metrik | Rumus | Keterangan |
|--------|-------|------------|
| **Accuracy** | (TP+TN)/(TP+TN+FP+FN) | Persentase prediksi benar dari total data |
| **Precision** | TP/(TP+FP) | Proporsi prediksi REAL yang benar-benar REAL |
| **Recall** | TP/(TP+FN) | Proporsi berita REAL yang berhasil terdeteksi |
| **AUC** | Area Under ROC Curve | Kemampuan model membedakan kedua kelas |

Keterangan:
- **TP** = True Positive (REAL diprediksi REAL)  
- **TN** = True Negative (FAKE diprediksi FAKE)  
- **FP** = False Positive (FAKE diprediksi REAL) \u2014 _false alarm_  
- **FN** = False Negative (REAL diprediksi FAKE) \u2014 _miss_

### 7.7 Evaluator

Mengevaluasi performa model menggunakan TensorFlow Model Analysis (TFMA) dan memberikan _blessing_ jika memenuhi threshold.

In [ ]:
eval_config = evaluator_pb2.EvalConfig(
    model_specs=[evaluator_pb2.ModelSpec(signature_name='serving_default')],
    slicing_specs=[evaluator_pb2.SlicingSpec()],
    metrics_specs=[
        evaluator_pb2.MetricsSpec(metrics=[
            evaluator_pb2.MetricConfig(class_name='BinaryAccuracy'),
            evaluator_pb2.MetricConfig(class_name='Precision'),
            evaluator_pb2.MetricConfig(class_name='Recall'),
            evaluator_pb2.MetricConfig(class_name='AUC'),
        ])
    ]
)

evaluator = Evaluator(
    examples=example_gen.outputs['examples'],
    model=trainer.outputs['model'],
    eval_config=eval_config
)
context.run(evaluator)

context.show(evaluator.outputs['evaluation'])

### 7.8 Resolver

Mendapatkan model terbaik yang sudah mendapatkan _blessing_ dari Evaluator.

In [ ]:
resolver = Resolver(
    strategy_class=LatestBlessedModelStrategy,
    model=Channel(type=Model),
    model_blessing=Channel(type=ModelBlessing)
).with_id('latest_blessed_model_resolver')

context.run(resolver)
print('Resolver executed. Latest blessed model resolved.')

### 7.9 Pusher

Menyimpan model yang telah di-bless ke folder `serving_model/` untuk deployment.

In [ ]:
pusher = Pusher(
    model=trainer.outputs['model'],
    model_blessing=evaluator.outputs['blessing'],
    push_destination=pusher_pb2.PushDestination(
        filesystem=pusher_pb2.PushDestination.Filesystem(
            base_directory=os.path.abspath('serving_model')
        )
    )
)
context.run(pusher)

print(f'Model pushed to: {os.path.abspath("serving_model")}')

## 10. Hasil Performa Model

Muat model yang sudah disimpan dan evaluasi performanya pada data uji.

In [ ]:
serving_model_path = os.path.abspath('serving_model')

if not os.path.exists(serving_model_path):
    model_artifact = trainer.outputs['model'].get()[0]
    serving_model_path = model_artifact.uri

subdirs = [d for d in os.listdir(serving_model_path)
           if os.path.isdir(os.path.join(serving_model_path, d))]
if subdirs:
    latest_model = os.path.join(serving_model_path, sorted(subdirs)[-1])
    loaded_model = tf.keras.models.load_model(latest_model)
    print(f'Model loaded from: {latest_model}')
else:
    loaded_model = tf.keras.models.load_model(serving_model_path)
    print(f'Model loaded from: {serving_model_path}')

loaded_model.summary()

In [ ]:
import math

eval_results = loaded_model.evaluate(
    df_eval['text'].values,
    (df_eval['label'] == 'REAL').astype(int).values,
    verbose=0
)

metrics_names = ['loss', 'accuracy', 'precision', 'recall', 'auc']

print('=' * 60)
print('    Model Performance on Evaluation Data')
print('=' * 60)
for name, value in zip(metrics_names, eval_results):
    print(f'  {name:12s}: {value:.4f}')
print('=' * 60)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

metrics_values = eval_results[1:]
metrics_labels = metrics_names[1:]
colors = ['#2ecc71', '#3498db', '#e74c3c', '#f39c12']

bars = axes[0].bar(metrics_labels, metrics_values, color=colors, width=0.6)
axes[0].set_ylim(0, 1)
axes[0].set_ylabel('Score')
axes[0].set_title('Model Evaluation Metrics', fontsize=14, fontweight='bold')
axes[0].axhline(y=0.9, color='gray', linestyle='--', alpha=0.7, label='90% threshold')
axes[0].legend()

for bar, val in zip(bars, metrics_values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                 f'{val:.4f}', ha='center', fontweight='bold', fontsize=11)

label_counts = df_eval['label'].value_counts()
wedges, texts, autotexts = axes[1].pie(
    label_counts.values, labels=label_counts.index,
    autopct='%1.1f%%', colors=['#e74c3c', '#2ecc71'],
    startangle=90, explode=(0.05, 0.05)
)
axes[1].set_title('Evaluation Data Label Distribution', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
sample_news = [
    "Scientists discover a new planet that could support human life in a nearby solar system.",
    "You won't believe what this celebrity did! Click here to find out the shocking truth.",
    "The government has announced new policies to address climate change and reduce carbon emissions.",
    "This miracle cure will change your life forever. Doctors hate this one simple trick!"
]

predictions = loaded_model.predict(np.array(sample_news), verbose=0)

print('Sample Predictions:')
print('=' * 80)
for i, (news, pred) in enumerate(zip(sample_news, predictions)):
    label = 'REAL' if pred[0] >= 0.5 else 'FAKE'
    confidence = pred[0] if pred[0] >= 0.5 else 1 - pred[0]
    print(f'[{i+1}] {news[:70]}...')
    print(f'    \u2192 {label} (confidence: {confidence:.4f})')
    print()

## 11. Kesimpulan

Proyek ini berhasil membangun **pipeline TFX end-to-end** untuk klasifikasi berita palsu:

- **9 komponen TFX** berhasil dijalankan secara berurutan: ExampleGen \u2192 StatisticsGen \u2192 SchemaGen \u2192 ExampleValidator \u2192 Transform \u2192 Trainer \u2192 Evaluator \u2192 Resolver \u2192 Pusher
- **Dataset** Fake and Real News (~44.000 sampel) berhasil diproses dan dibagi menjadi train (80%) dan eval (20%)
- **Arsitektur model** menggunakan TextVectorization + Embedding + GlobalAveragePooling1D + Dense + Dropout
- **Model** disimpan ke folder `serving_model/` melalui komponen Pusher, siap untuk deployment
- **Pipeline** dijalankan menggunakan InteractiveContext sehingga setiap langkah dapat dimonitor

Pipeline ini dapat dikembangkan lebih lanjut ke production dengan:
- Menggunakan **Orchestrator** seperti Apache Beam, Kubeflow, atau Vertex AI Pipelines
- Menambahkan **CI/CD** untuk otomatisasi training dan deployment
- Deploy model sebagai **REST API** menggunakan TensorFlow Serving